<a href="https://colab.research.google.com/github/chyoon114/CAS2105-HW6/blob/main/CAS2105_HW6_Chaeyoon_Kim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Dataset Loading and Preprocessing

In [ ]:
# 1. Install gdown library (for Google Drive file download)

!pip install gdown

In [ ]:
# 2. Setup & Load Data

import pandas as pd
import gdown # Add gdown library
import os # Add os module for file path management

raw_file_id = '1Seq2rZcO7stcKLcjiQTuOGeGtFVV7GyW' #  Google Drive ID of running_belt_reviews_raw.csv
file_path = "sample_data/running_belt_reviews_raw.csv"

# Create sample_data directory if it does not exist
if not os.path.exists('sample_data'):
    os.makedirs('sample_data')

# Download the file from Google Drive
print(f"Downloading raw data from Google Drive to {file_path}...")

gdown.download(id=raw_file_id, output=file_path, quiet=False) # Display progress with quiet=False
print("Download complete.")

df = pd.read_csv(file_path, encoding='cp949')
print(f"Successfully loaded raw data. Total rows: {len(df)}")

print("\n--- DataFrame Preview after loading ---")
display(df.head())

Downloading...
From: https://drive.google.com/uc?id=1Seq2rZcO7stcKLcjiQTuOGeGtFVV7GyW
To: /content/sample_data/running_belt_reviews_raw.csv
100%|██████████| 869k/869k [00:00<00:00, 73.0MB/s]


Download complete.
Successfully loaded raw data. Total rows: 5880

--- DataFrame Preview after loading ---


,purchase_id,review_date,product_option,review_text,rating,quantity
0,remi***,2025-11-30T02:35:36.215+00:00,상품선택: 러닝벨트 / 색상: 블랙 / 사이즈: M,매우 만족합니다. 핸드폰 무게 느껴지지 않을 정도로 편하네요..,5,1
1,7803***,2025-11-01T11:21:51.584+00:00,상품선택: 러닝벨트 / 색상: 블랙 / 사이즈: S,잘 쓰고 다니는데요. 달리다가 핸드폰이 한 번 탈출한 적이 있어요. 잘 넣은 거 같...,4,1
2,hyek***,2025-11-28T08:56:54.076+00:00,상품선택: 러닝벨트 / 색상: 블랙 / 사이즈: S,짱짱하니 좋아요 굿,5,1
3,bdw8*****,2025-11-23T16:08:58.890+00:00,상품선택: 러닝벨트 / 색상: 블랙 / 사이즈: M,178에 64키로 m사이즈는 아주 쪼금 헐렁하지만 맞긴한데 s가 좋을듯합니다,5,1
4,jjsl****,2025-11-22T09:41:17.006+00:00,상품선택: 러닝벨트 / 색상: 블랙 / 사이즈: S,아주주주 만족만족 잘쓰고있음,5,1


In [ ]:
# 3. Select Columns

# We need 'review_text', 'rating'.
target_columns = ['review_text', 'rating']
df = df[target_columns]

In [ ]:
# 4. Filtering & Labeling

# Logic:
# - Score 5       -> Positive (Label 1)
# - Score 1 or 2  -> Negative (Label 0)
# - Score 3 or 4  -> Excluded (to ensure clear sentiment distinction)

positive_df = df[df['rating'] == 5].copy()
negative_df = df[df['rating'] <= 2].copy()

# Assign labels
positive_df['label'] = 1
negative_df['label'] = 0

print(f" - Candidates for Positive (5 stars): {len(positive_df)}")
print(f" - Candidates for Negative (1-2 stars): {len(negative_df)}")

 - Candidates for Positive (5 stars): 4752
 - Candidates for Negative (1-2 stars): 52


In [ ]:
# 5. Balancing the Dataset (Random Sampling)

# We aim for a balanced dataset (50 Positive, 50 Negative).
samples_per_class = 50

# Sample randomly. If fewer than 50 examples exist, take all of them.
final_pos = positive_df.sample(n=min(len(positive_df), samples_per_class), random_state=42)
final_neg = negative_df.sample(n=min(len(negative_df), samples_per_class), random_state=42)

In [ ]:
# 6. Merge and Format

final_df = pd.concat([final_pos, final_neg])

# Keep only 'text' and 'label' columns
final_df = final_df[['review_text', 'label']]

# Shuffle the final dataset
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# 7. Save to CSV

output_filename = "sample_data/running_belt_reviews_processed.csv"
final_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print("="*40)
print(f"Processing Complete! Saved as '{output_filename}'")
print("Final Dataset Distribution:")
print(final_df['label'].value_counts())
print("="*40)
print("\nPreview of the first 5 rows:")
print(final_df.head())

Processing Complete! Saved as 'sample_data/running_belt_reviews_processed.csv'
Final Dataset Distribution:
label
0    50
1    50
Name: count, dtype: int64

Preview of the first 5 rows:
                                         review_text  label
0  갤노트8쓰는데 자꾸 핸드폰이 빠져요 ㅠ 한번도 안찍어 먹은건데 잠금 단추라도 손수 ...      0
1                           개인 체격에 맞게 주문하여야 할 꺼 갇습니다      0
2                                              불친절해요      0
3  다른 제품이랑 큰 차이 없네요. 비싸게 주고 사지마시고 이거 사서 쓰세요 다들. 좋...      1
4                                   달리기할때 디게유용하구좋아요!      1


# 2. Naive Baseline: Keyword-Based Text Classification

In [ ]:
# 1. Setup & Load Data

import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
import os

# Processed file is now generated locally by previous steps, so we load it from there.
file_path = "sample_data/running_belt_reviews_processed.csv"

# Load the dataset
df = pd.read_csv(file_path)
print("Dataset loaded successfully.")
print(f"Total reviews: {len(df)}")

Dataset loaded successfully.
Total reviews: 100


In [ ]:
# 2. Define Naïve Baseline (Keyword Rules)

def naive_baseline(text):
    """
    Classifies sentiment based on the presence of negative keywords.
    - Input: Review text
    - Output: 0 (Negative) if a keyword is found, else 1 (Positive)
    """
    # Negative keywords specific to 'Running Belts'
    # These are the "Rules" we are testing.
    negative_keywords = [
        '불편', '고장', '최악', '덜렁', '냄새', '아파', '떨어',
        '작아', '엉망', '느리', '별로', '환불', '흔들', '반품',
        '실망', '무거', '약해', '비싸', '안'
    ]
    # '안' (not) is tricky. It might catch "안 흔들려요" (Good) as Negative.

    for keyword in negative_keywords:
        if keyword in text:
            return 0  # Predict Negative
    return 1  # Predict Positive (Default)

# Apply the baseline
print("Running Naïve Baseline Classifier...")
df['pred_baseline'] = df['review_text'].apply(naive_baseline)

Running Naïve Baseline Classifier...


In [ ]:
# 3. Evaluate Performance

# Compare our 'Rules' against the 'True Labels' (Review Score)
accuracy = accuracy_score(df['label'], df['pred_baseline'])

print(f"BASELINE ACCURACY: {accuracy:.4f}")

BASELINE ACCURACY: 0.5400


In [ ]:
# 4. Analyze Errors

# Show cases where the Rule was WRONG
print("Failure Cases (Where the Rule failed):")
wrong_cases = df[df['label'] != df['pred_baseline']]

if not wrong_cases.empty:
    for index, row in wrong_cases.head(5).iterrows():
        print(f"\n[Review]: {row['review_text']}") # Corrected from row['text']
        print(f" - True Label: {row['label']} ({'Positive' if row['label']==1 else 'Negative'})")
        print(f" - Our Prediction: {row['pred_baseline']} (Wrong)")
else:
    print("The rule got everything right (or the dataset is too small).")

Failure Cases (Where the Rule failed):

[Review]: 개인 체격에 맞게 주문하여야 할 꺼 갇습니다
 - True Label: 0 (Negative)
 - Our Prediction: 1 (Wrong)

[Review]: 불친절해요
 - True Label: 0 (Negative)
 - Our Prediction: 1 (Wrong)

[Review]: 다른 제품이랑 큰 차이 없네요. 비싸게 주고 사지마시고 이거 사서 쓰세요 다들. 좋네요. 잘 흘러냐리지도 않고 물통도 넣어서 다닐 수 있습니다
 - True Label: 1 (Positive)
 - Our Prediction: 0 (Wrong)

[Review]: 러닝할 때 핸드폰을 어떻게 보관할지 고민했었는데 안성맞춤인 제품을 찾아서 너무 좋아요
하나도 불편하지 않고 딱 고정되서 너무 좋네요!
잘 쓸게요~~
 - True Label: 1 (Positive)
 - Our Prediction: 0 (Wrong)

[Review]: 허리 32인치에 M사이즈 맞구요. 생각보다 두껍고 바느질 일부 누락되었네요. 휴대폰 넣을때 튿어지지 않게 조심해서 착용해야겠네요.
 - True Label: 0 (Negative)
 - Our Prediction: 1 (Wrong)


# 3. AI Pipeline

In [ ]:
# 1. Load Model: matthewburke/korean_sentiment

from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="matthewburke/korean_sentiment",
    top_k=None
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/887 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
# 2. Define Inference Function

def ai_pipeline(text):
    """
    Classifies sentiment using the AI model.
    - Input: Review text
    - Output: 0 (Negative) or 1 (Positive)
    """
    # Run the model
    results = classifier(text, truncation=True, max_length=512)

    # Sort results by score to get the top prediction
    top_result = sorted(results[0], key=lambda x: x['score'], reverse=True)[0]

    # The 'matthewburke' model outputs 'LABEL_1' for positive and 'LABEL_0' for negative.
    if top_result['label'] == 'LABEL_1':
        return 1 # Positive
    else:
        return 0 # Negative (LABEL_0)

In [ ]:
# 3. Run Inference

if 'df' in locals() and 'review_text' in df.columns:
    print("AI is reading reviews...")
    df['pred_ai'] = df['review_text'].apply(ai_pipeline)
    print("AI Prediction Complete!")
else:
    print("Error: Dataframe 'df' or 'review_text' column not found. Please ensure previous steps were run.")

AI is reading reviews...
AI Prediction Complete!


In [ ]:
# 4. Compare Results

if 'df' in locals() and 'label' in df.columns and 'pred_baseline' in df.columns and 'pred_ai' in df.columns:
    acc_baseline = accuracy_score(df['label'], df['pred_baseline'])
    acc_ai = accuracy_score(df['label'], df['pred_ai'])

    print("="*50)
    print("FINAL RESULTS")
    print("="*50)
    print(f"1. Naïve Baseline : {acc_baseline:.2f}")
    print(f"2. AI Pipeline    : {acc_ai:.2f}")
else:
    print("Warning: Required columns ('label', 'pred_baseline', 'pred_ai') missing. Cannot calculate accuracy comparison.")

FINAL RESULTS
1. Naïve Baseline : 0.54
2. AI Pipeline    : 0.86


In [ ]:
# 5. Find Success Cases (Insight)

if 'df' in locals() and 'label' in df.columns and 'pred_baseline' in df.columns and 'pred_ai' in df.columns:
    improvement_cases = df[(df['label'] != df['pred_baseline']) & (df['label'] == df['pred_ai'])]

    if not improvement_cases.empty:
        print("[Insight] AI Success Cases (Better than Baseline):")
        for i, row in improvement_cases.head(3).iterrows():
            print(f" - Review: {row['review_text']}")
            print(f" - AI Prediction: Correct ({row['pred_ai']})")
    else:
        print("No specific improvement cases found in this batch.")
else:
    print("Warning: Required columns ('label', 'pred_baseline', 'pred_ai') missing. Cannot analyze improvement cases.")

[Insight] AI Success Cases (Better than Baseline):
 - Review: 개인 체격에 맞게 주문하여야 할 꺼 갇습니다
 - AI Prediction: Correct (0)
 - Review: 불친절해요
 - AI Prediction: Correct (0)
 - Review: 다른 제품이랑 큰 차이 없네요. 비싸게 주고 사지마시고 이거 사서 쓰세요 다들. 좋네요. 잘 흘러냐리지도 않고 물통도 넣어서 다닐 수 있습니다
 - AI Prediction: Correct (1)


# 4.  Export Full Results for Analysis

In [ ]:
# 1. Create a detailed analysis dataframe

# We will add a 'Category' column to easily filter in Excel
analysis_df = df.copy()

def classify_result(row):
    baseline_correct = (row['pred_baseline'] == row['label'])
    ai_correct = (row['pred_ai'] == row['label'])

    if ai_correct and not baseline_correct:
        return "AI Wins (AI Success)"
    elif not ai_correct and baseline_correct:
        return "Baseline Wins (AI Fail)"
    elif ai_correct and baseline_correct:
        return "Both Correct"
    else:
        return "Both Wrong"

analysis_df['Analysis_Category'] = analysis_df.apply(classify_result, axis=1)

In [ ]:
# 2. Reorder columns for better readability

# Ensure column names exist
cols = ['review_text', 'label', 'pred_baseline', 'pred_ai', 'Analysis_Category']
final_export = analysis_df[cols]

In [ ]:
# 3. Save to CSV

save_filename = "running_belt_full_analysis.csv"
final_export.to_csv(save_filename, index=False, encoding='utf-8-sig')

print("="*60)
print(f"Full analysis file saved: '{save_filename}'")
print("="*60)
print("\n[Summary Statistics]")
print(final_export['Analysis_Category'].value_counts())
print("\nDownload the file from the left panel and check it in Excel.")

Full analysis file saved: 'running_belt_full_analysis.csv'

[Summary Statistics]
Analysis_Category
Both Correct               49
AI Wins (AI Success)       37
Both Wrong                  9
Baseline Wins (AI Fail)     5
Name: count, dtype: int64

Download the file from the left panel and check it in Excel.
